# 🛠️ Making Your Own Module — The First Box of Your Own Library

Welcome to the second topic of this unit: creating modules. So far you have been a *consumer* of Python's standard library — importing `math`, `random`, and `datetime` built by other people. Now the tables turn: you become a **producer**. You will write your own `.py` file, import it, inspect it, and control how much of it runs. This is the step where “Python programming” becomes “building software that other code can reuse.”

## 1. What you will learn in this notebook

1. **A module is just a file** — a `.py` file full of functions, classes, and variables. Writing one is no different from writing any script you have done so far; what changes is how Python finds and loads it.
2. **`sys.path` — Python's map to every module.** When you `import grade_utils`, Python searches `sys.path`, an ordered list of folders. If your file's folder is on that list, the import succeeds. We will insert a temp folder into `sys.path` live.
3. **`dir()` and docstrings — your module's self-description.** Every module can announce its own contents and explain its own functions.
4. **The `__name__ == "__main__"` guard — the two lives of a file.** The same file can behave as a quiet import library or as a loud standalone script, depending on how it is launched.
5. **`sys.modules` cache and `__pycache__`** — behind-the-scenes machinery that explains why re-imports are cheap and why old code sometimes “sticks.”

## 2. The big idea

Real projects reuse helper functions everywhere: grading marks, formatting dates, fetching data. Copy-pasting the same code into ten notebooks is a recipe for bugs — fix the logic in one place and nine copies stay wrong. The professional move is to write the helper once inside a module and `import` it wherever it is needed. That one decision turns scattered scripts into a maintainable library.

## 3. How the demos work

Because a notebook cannot easily create files next to itself, the demos write modules into temporary folders via `tempfile` + `sys.path.insert(0, tmp)`. This keeps your real workspace clean while letting you experience the full import cycle: Python finds the file, executes it bottom-to-top, and returns a module object whose names you can use immediately. Run each cell, watch the output, and notice how ordinary a “library” really is.

## 🍱 Real-World Hook: One Recipe Book for Every Kitchen

After two launched projects, you notice a pattern: you are writing the same helper functions again and again — averaging marks, mapping scores to grades, picking the topper. Photocopying those helpers into every new notebook works for a while, but every bugfix now has to be repeated in every copy.

The smart idea: one **recipe book** — a single `grade_utils.py` that every counter imports. Fix the recipe once; every kitchen benefits. That is the entire philosophy of modules: write code once, reuse it with an import, treat your own file with the same respect you give the standard library.

### 📖 Definitions (interview-grade)

> **module** — any `.py` file that can be imported.
> **import cycle** — 1) look in the `sys.modules` cache; 2) scan the folders in `sys.path`; 3) found → execute the file → bind the namespace.

The first cell below walks you through this exact cycle in a few lines of code.

# 1. Your First Module — Write, Point, Import

## 1.1 A module is just a `.py` file

There is nothing magical about a module. It is a plain Python file that defines functions, classes, and variables at the top level. When you run `import grade_utils`, the interpreter looks for a file called `grade_utils.py` in the folders listed in `sys.path`. If it is found, Python executes the file once, collects every top-level name into a module object, and binds that object to the name `grade_utils`. If the file is not found, you get `ModuleNotFoundError`.

The demo writes a real module with `open()`. It contains three functions: `average(marks_list)` computes `round(sum / len, 1)`, `grade_letter(score)` maps numbers to A/B/C/D/F through `if` thresholds, and `topper(students)` returns the name with the highest score using `max(students, key=students.get)`. That `key=` trick is worth remembering: `max` compares dictionary *values* when you give it the `get` method as the key function.

Then comes the punctuation mark: `sys.path.insert(0, tmp)`. The temp folder now leads Python's search list, so the freshly written file becomes importable. The alias `import grade_utils as gu` shortens the namespace, and now every function is one dot away: `gu.average(...)`, `gu.grade_letter(...)`, `gu.topper(...)`.

In [1]:
# 🐣 STEP-1: Apna grade_utils.py likho + import karo! (tempdir-magic-live ✨)
import sys, os, tempfile

MODULE_CODE = '''"""grade_utils — Student-marks ke helper-functions ki library! 📚"""

def average(marks_list):
    """Marks-list ka average nikaalo (round-1!)."""
    return round(sum(marks_list) / len(marks_list), 1)

def grade_letter(score):
    """90:A, 75:B, 60:C, 40:D, <40:F — exam-card style!"""
    if score >= 90: return "A"
    if score >= 75: return "B"
    if score >= 60: return "C"
    if score >= 40: return "D"
    return "F"

def topper(students):
    """{'name': marks} dict me se sabse-badiya-student name!"""
    return max(students, key=students.get)
'''

tmp = tempfile.mkdtemp()
with open(os.path.join(tmp, "grade_utils.py"), "w") as f:
    f.write(MODULE_CODE)

sys.path.insert(0, tmp)              # python ko BATO: "yahan bhi dhekh!" 🗺️
import grade_utils as gu             # 🎉 private-library live!

print("average:", gu.average([80, 90, 70]))
print("grade 88:", gu.grade_letter(88))
print("topper  :", gu.topper({"Sana": 92, "Amit": 85, "Priya": 96}))
# Expected OUTPUT:
# average: 80.0
# grade 88: B
# topper  : Priya

average: 80.0
grade 88: B
topper  : Priya


## 1.2 Module peeping — `dir()` and docstrings

A module should be able to explain itself. The built-in `dir(module)` lists every name the module exposes — functions, variables, and also dunder machinery like `__name__` and `__doc__`. The demo filters those out with `[n for n in dir(gu) if not n.startswith("_")]`, leaving exactly the public surface: `['average', 'grade_letter', 'topper']`. That is a perfect sanity check: the module shows only what we designed it to show.

The second self-explanatory tool is the **docstring**. The triple-quoted comment at the top of the file becomes `gu.__doc__` — the module's own one-line biography. Each function's docstring is equally queryable: `gu.average.__doc__` prints what `average` promises to do. In an interactive console, `help(gu.average)` renders the signature and docstring into a tidy card.

Why does this matter at scale? Because documentation that lives next to the code cannot rot: it is regenerated from the file itself. When you revisit a module months later, `__doc__` and `dir()` give you an instant, accurate map.

In [2]:
# 🔍 STEP-2: Module-peeping — dir() + docstring-self-guide! 📖
peek = [n for n in dir(gu) if not n.startswith("_")]
print("public-tools:", peek)
print("doc         :", gu.__doc__)

# tip: console me help(gu.average) likho — full-card milega! 💡
func_doc = gu.average.__doc__
print("average-doc :", func_doc)
# Expected OUTPUT:
# public-tools: ['average', 'grade_letter', 'topper']
# doc         :	grade_utils — Student-marks ke helper-functions ki library! 📚
# average-doc :	Marks-list ka average nikaalo (round-1!).

public-tools: ['average', 'grade_letter', 'topper']
doc         : grade_utils — Student-marks ke helper-functions ki library! 📚
average-doc : Marks-list ka average nikaalo (round-1!).


## 1.3 The import cache — `sys.modules`

Run the same import twice and you might expect the file to execute twice. It does not. Python keeps every loaded module in a dictionary called `sys.modules`, keyed by the module's name. The first `import grade_utils` executes the file and stores the result. A second `import grade_utils as gu_2` finds the name already cached and returns the *same* object without re-reading the file — hence `gu is gu_2` prints `True`.

This cache is a performance feature: importing a heavy module (imagine hundreds of functions) would otherwise be catastrophically slow. But it is also the classic footgun in notebooks: after you edit your `.py` file on disk, re-importing still gives you the stale cached version. The official cure is `importlib.reload(module)`, or simply restarting the kernel. The rule to internalise: *a module executes exactly once per process* — the cache is your friend until you are actively editing, and then it feels like an enemy.

In [3]:
# 🗄️ STEP-3: sys.modules cache — "already-load hai ji!" proof!
print("in-cache?   :", "grade_utils" in sys.modules)

# dobara-import: NAYA-execute NAHI — cache-se le-te! (speed-priya!)
import grade_utils as gu_2
print("same-object?  :", gu is gu_2)
print("(cache-rule → import 2nd-time CHEAP; test-fixes require reload!)")
# Expected OUTPUT:
# in-cache?   : True
# same-object?  : True
# (cache-rule → import 2nd-time CHEAP; test-fixes require reload!)

in-cache?   : True
same-object?  : True
(cache-rule → import 2nd-time CHEAP; test-fixes require reload!)


# 2. The Two Lives of a File

## 2.1 Import-silent vs script-loud

Every Python file has two possible lives. If you run `python myfile.py` in a terminal, the file is a **script**: its top-to-bottom code executes, `prints` fire, and it behaves like a program. If you `import myfile` from elsewhere, the file is a **module**: functions and classes load, but you usually do not want its demo prints and test blocks cluttering your session.

How does Python know which life a file is living? It sets a magic variable, `__name__`:

- when the file is executed directly → `__name__ == "__main__"` 🎬
- when the file is imported → `__name__ == "file's own name"` (e.g. `"loud_fetcher"`)

The idiom `if __name__ == "__main__":` therefore arms a block that fires only in script mode. The next cell builds a module with an unguarded loud print (“loud-banner-at-import”) and a guarded print (“script-mode special-lines”). On import, only the banner fires; the guarded line stays silent — exactly the professional behaviour you want from a library.

In [4]:
# 🔑 STEP-4: __name__ == "__main__" guard — TVN-democracy-scene!
# Scene: module ke andar loud-prints 👇
LOUD_MODULE = '''DB = {"sana": 90}

def fetch(name):
    return DB[name]

print("loud-banner-at-import 🎺")

if __name__ == "__main__":
    print("script-mode special-lines! 🎬")
'''

tmp2 = tempfile.mkdtemp()
script_path = os.path.join(tmp2, "loud_fetcher.py")
with open(script_path, "w") as f:
    f.write(LOUD_MODULE)

sys.path.insert(0, tmp2)
print("=== IMPORT-side ===")
import loud_fetcher                                   # sirf print-1 chalega!
print("fetch-result:", loud_fetcher.fetch("sana"))
# Expected OUTPUT:
# === IMPORT-side ===
# loud-banner-at-import 🎺
# fetch-result: 90

=== IMPORT-side ===
loud-banner-at-import 🎺
fetch-result: 90


## 2.2 Running the same file as a script — `subprocess`

The previous cell proved what *import-time* silence looks like. This cell proves the sunny side: the same file, launched directly, becomes chatty. To launch it, we need a brand-new Python process — not the notebook's kernel — so the demo uses `subprocess.run([sys.executable, script_path], capture_output=True, text=True)`.

`sys.executable` is the full path of the Python interpreter currently running the notebook — passing it guarantees we use the very same Python. `capture_output=True` captures the child's stdout into `result.stdout`, and `text=True` decodes it as text (instead of bytes). `result.stdout.strip()` then gives us exactly what a terminal user would have seen: both the banner AND “script-mode special-lines!”, because in script mode `__name__` is `"__main__"`.

The contrast between the two cells is the whole lesson: **the guard makes one file behave as a quiet library when imported and as a loud program when executed.** Finally, the cleanup (`sys.path.pop(0)` twice, plus evicting both modules from `sys.modules`) restores the environment so later cells start fresh.

In [5]:
# 🎬 STEP-5: SAME-file ko SCRIPT-mode run (subprocess!) — guard-line fires!
import subprocess
result = subprocess.run([sys.executable, script_path],
                        capture_output=True, text=True)
print("=== SCRIPT-side ===")
print(result.stdout.strip())
print("(notice: dono-prints yahan! — import-side pe sirf-eka-PRINT!) 🔑")

# cleanup-ka pack
sys.path.pop(0); sys.path.pop(0)
sys.modules.pop("grade_utils", None); sys.modules.pop("loud_fetcher", None)
# Expected OUTPUT:
# === SCRIPT-side ===
# loud-banner-at-import 🎺
# script-mode special-lines! 🎬
# (notice: dono-prints yahan! — import-side pe sirf-eka-PRINT!) 🔑

=== SCRIPT-side ===
loud-banner-at-import 🎺
script-mode special-lines! 🎬
(notice: dono-prints yahan! — import-side pe sirf-eka-PRINT!) 🔑


<module 'loud_fetcher' from '/tmp/tmp4mbk0twp/loud_fetcher.py'>

### 🔑 The Meaning of the Guard (the golden rule)

```python
if __name__ == "__main__":
    # these lines run ONLY when the file is executed directly
    # on the import side they are skipped — the module stays a silent library
```

- running `python file.py` → Python sets `__name__` to the special string `"__main__"` 🎬
- running `import file` → Python sets `__name__` to the module's name, `"file"` (silent import!)

> **Rule to write down:** put demo code, self-tests, and “run-me” examples inside the guard block. That way your module stays quiet and polite when somebody imports it, yet fully alive when launched directly. Every serious Python file you will read in the wild obeys this rule.

## ⚠️ 5 Common Mistakes

| # | Mistake | Correct habit |
|---|---|---|
| 1 | unguarded loud prints in a module file | protect with `if __name__ == "__main__":` 🔑 |
| 2 | leaving temp folders crammed into `sys.path` | after the demo pop them out + evict from `sys.modules` 🧹 |
| 3 | "I edited my file but the old result keeps coming back!" | the import cache! use `importlib.reload(module)` 🔄 |
| 4 | scattering imports deep inside functions | keep imports at the top of the file (PEP 8) — unless genuinely needed at runtime |
| 5 | filenames with spaces, dashes, or capitals | lowercase `snake_case` is the Python way 📁 |

> **Cache principle:** importing the same module a second time is nearly free (the cache serves it); the first import executes the whole file. The guard is what keeps that one-time execution from being noisy.

## Summary

You are no longer only a user of modules — you are their author. A module is simply a `.py` file of functions, classes, and variables, and importing it is a 3-step journey: check `sys.modules` for a cached copy, scan `sys.path` for the file, then execute it once and bind its namespace. We built and imported `grade_utils.py` live, peered inside with `dir()` and docstrings, and proved that a second import is cheap because the cache reuses the same object. The star of the show was the `__name__ == "__main__"` guard: it gives one file two lives — a silent library when imported, a loud script when run directly. Finally, clean `sys.path` and `sys.modules` after demos, and never recreate standard-library names like `math.py`. Next stop: packages — folders of modules with an `__init__.py` heart. 🏪